# Notebook 02 — RAG Pipeline

Demonstrates retrieval-augmented generation against the sample program corpus,
including retrieval failure modes and their mitigations.

<!-- TODO main-session: expand teaching framing; tie back to NB 01 closing arc -->

## Setup

Adds the repo root to `sys.path`, loads environment variables, and imports the public RAG API.

<!-- TODO main-session: expand teaching framing -->

In [ ]:
from __future__ import annotations
import os, sys, logging
from pathlib import Path

# Silence ChromaDB telemetry noise (posthog API mismatch; does not affect functionality)
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env", override=False)

from src.rag import ingest, retrieve, RetrievedDocument
from src.llm import LLMClient

provider = os.getenv("LLM_PROVIDER", "anthropic")
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Provider: {provider} · Anthropic key present: {has_key}")


Provider: anthropic · Anthropic key present: True


## Ingest the corpus

Loads the five sample markdown documents, chunks them, embeds them, and persists the vector store.

<!-- TODO main-session: expand teaching framing -->

In [2]:
corpus_dir = repo_root / "data"
persist_dir = repo_root / "data" / "chroma_nb02"

result = ingest(
    corpus_dir=corpus_dir,
    persist_dir=persist_dir,
    chunk_size=500,
    chunk_overlap=50,
)

print(f"documents_loaded  : {result.documents_loaded}")
print(f"chunks_created    : {result.chunks_created}")
print(f"chunks_indexed    : {result.chunks_indexed}")
print(f"vector_store_path : {result.vector_store_path}")
print(f"embedding_model   : {result.embedding_model}")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 7f491bb3-9b7e-4982-84f0-07180fa2e298)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


documents_loaded  : 5
chunks_created    : 42
chunks_indexed    : 42
vector_store_path : c:\Users\narla\OneDrive\Desktop\TalentSprint\IISc_GenAI_C2\LLMOps\llmops-session\data\chroma_nb02
embedding_model   : sentence-transformers/all-MiniLM-L6-v2


## Baseline retrieval

Retrieve the top-5 chunks for a simple policy question and inspect scores + source priority.

<!-- TODO main-session: expand teaching framing -->


In [4]:
hits = retrieve(persist_dir, "What is the late submission policy?", k=5)

SEP = "-" * 60
for i, doc in enumerate(hits, 1):
    doc_id = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    print(f"Hit {i}")
    print(f"  document_id     : {doc_id}")
    print(f"  source_priority : {priority}")
    print(f"  score           : {doc.score:.4f}")
    print(f"  text (first 200): {doc.chunk.text[:200]!r}")
    print(SEP)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Hit 1
  document_id     : program_policy
  source_priority : 1
  score           : 0.9911
  text (first 200): '- Submissions up to **48 hours late** receive full credit with no\n  penalty if a brief note is added to the submission explaining the delay.\n- Submissions **48–168 hours late** (i.e. up to one week) r'
------------------------------------------------------------
Hit 2
  document_id     : assignment_guidelines
  source_priority : 2
  score           : 0.9769
  text (first 200): '## Late Submission Handling\n\nSee the **Program Policy** document, section *Late Submission Policy*, for\nthe authoritative rules. In summary: 48 hours grace with note → 10% penalty\nup to one week → not'
------------------------------------------------------------
Hit 3
  document_id     : faq
  source_priority : 5
  score           : 0.9441
  text (first 200): '**Q: What file format?**\nWhatever the assignment brief specifies. Default is `.ipynb` with outputs\nsaved for notebook assignments.\n\n**Q

## Notice the source_priority

The policy doc (`source_priority=1`) ranks at or near the top for a policy question — the retriever naturally surfaces the authoritative source. Later sections show when this breaks down.

<!-- TODO main-session: expand teaching framing -->
